In [1]:
import pandas as pd
import requests
import time

In [2]:
# Cell 2 — load your cleaned dataset
combined = pd.read_csv('../data/rentora_model_ready.csv')  # adjust filename to whatever you actually saved

C:\Users\KUNDAN KUMAR\AppData\Local\Temp\ipykernel_30396\3108402289.py:2: DtypeWarning: Columns (0: size_sqft) have mixed types. Specify dtype option on import or set low_memory=False.
  combined = pd.read_csv('../data/rentora_model_ready.csv')  # adjust filename to whatever you actually saved


In [ ]:
# Cell 3 — get unique locations to query (don't repeat lookups for duplicate coordinates)
unique_locs = combined[['city', 'locality', 'latitude', 'longitude']].drop_duplicates(subset=['latitude', 'longitude'])

print(len(unique_locs))

7765


In [6]:
#Step-2:Overpass API query function — checks for highway/mall/river/mountain within a radius
import requests
import time

def get_geo_features(lat, lon, radius=2000):
    query = f"""
    [out:json][timeout:15];
    (
      way["highway"~"trunk|primary|motorway"](around:{radius},{lat},{lon});
    );
    out count;
    (
      node["shop"="mall"](around:{radius},{lat},{lon});
    );
    out count;
    (
      way["natural"="water"](around:{radius},{lat},{lon});
    );
    out count;
    (
      node["natural"="peak"](around:{radius},{lat},{lon});
    );
    out count;
    """
    try:
        response = requests.post("https://overpass-api.de/api/interpreter", data=query, timeout=20)
        data = response.json()
        counts = [int(el['tags'].get('total', 0)) for el in data['elements'] if el.get('type') == 'count']
        return pd.Series({
            'near_highway': counts[0] > 0 if len(counts) > 0 else None,
            'near_mall': counts[1] > 0 if len(counts) > 1 else None,
            'near_river': counts[2] > 0 if len(counts) > 2 else None,
            'near_mountain': counts[3] > 0 if len(counts) > 3 else None,
        })
    except Exception as e:
        print(f"Failed at {lat},{lon}: {e}")
        return pd.Series({'near_highway': None, 'near_mall': None, 'near_river': None, 'near_mountain': None})

In [7]:
#Step 3: Test on 5 rows first
test = unique_locs.head(5).copy()
results = test.apply(lambda r: get_geo_features(r['latitude'], r['longitude']), axis=1)
test = pd.concat([test, results], axis=1)
print(test)

Failed at 23.0445921,72.517344: Expecting value: line 1 column 1 (char 0)
Failed at 23.0260111,72.5567179: Expecting value: line 1 column 1 (char 0)
Failed at 23.0169268,72.5204317: Expecting value: line 1 column 1 (char 0)
Failed at 23.0238876,72.3851475: Expecting value: line 1 column 1 (char 0)
Failed at 23.0359998,72.5643429: Expecting value: line 1 column 1 (char 0)
        city     locality   latitude  longitude near_highway near_mall  \
0  Ahmedabad     Bodakdev  23.044592  72.517344         None      None   
1  Ahmedabad      CG Road  23.026011  72.556718         None      None   
2  Ahmedabad      Jodhpur  23.016927  72.520432         None      None   
3  Ahmedabad       Sanand  23.023888  72.385148         None      None   
4  Ahmedabad  Navrangpura  23.036000  72.564343         None      None   

  near_river near_mountain  
0       None          None  
1       None          None  
2       None          None  
3       None          None  
4       None          None  
